# 第18章 人工智能 - Part 1: 图与图遍历
# Chapter 18 AI - Part 1: Graphs and Graph Traversal

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

| 编号 | 目标 | 关键术语 |
|------|------|----------|
| 1 | 理解图的基本概念和分类 | Graph, Directed, Undirected, Weighted |
| 2 | 掌握图的两种存储表示 | Adjacency Matrix, Adjacency List |
| 3 | 实现深度优先和广度优先遍历 | DFS, BFS |
| 4 | 理解并实现Dijkstra最短路径算法 | Dijkstra's Algorithm |
| 5 | 了解A*算法及启发式搜索 | A* Algorithm, Heuristic |

---

## 1. 图(Graph)的概念

### 1.1 什么是图？

**图(Graph)** 是由**顶点(Vertices/Nodes)**和**边(Edges)**组成的数据结构。

数学定义：$G = (V, E)$，其中 $V$ 是顶点集合，$E$ 是边集合。

### 为什么图如此重要？

现实世界中大量问题可以建模为图：
- **社交网络**：人是顶点，好友关系是边
- **地图导航**：路口是顶点，道路是边，距离是权重
- **互联网**：网页是顶点，超链接是边
- **电路设计**：元件是顶点，导线是边

图之所以重要，是因为它是最**通用**的数据结构之一——树是特殊的图（无环连通图），链表是特殊的树。

### 1.2 图的分类

| 类型 | 英文 | 特点 | 例子 |
|------|------|------|------|
| 无向图 | Undirected Graph | 边没有方向 | 好友关系（互相的）|
| 有向图 | Directed Graph | 边有方向 | 微博关注（单向的）|
| 加权图 | Weighted Graph | 边有权重/代价 | 地图上的距离 |
| 无权图 | Unweighted Graph | 边没有权重 | 是否有连接 |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import deque
import heapq

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False



In [ ]:
# === 可视化：三种图的对比 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 顶点位置
positions = {
    'A': (0.2, 0.8), 'B': (0.8, 0.8),
    'C': (0.5, 0.5), 'D': (0.2, 0.2), 'E': (0.8, 0.2)
}

def draw_node(ax, name, pos, color='#4ECDC4'):
    circle = plt.Circle(pos, 0.08, color=color, ec='black', lw=2, zorder=5)
    ax.add_patch(circle)
    ax.text(pos[0], pos[1], name, ha='center', va='center',
            fontsize=14, fontweight='bold', zorder=6)

def draw_edge(ax, p1, p2, directed=False, weight=None, color='#333333'):
    if directed:
        ax.annotate('', xy=p2, xytext=p1,
            arrowprops=dict(arrowstyle='->', color=color, lw=2))
    else:
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=color, lw=2, zorder=1)
    if weight is not None:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my+0.05, str(weight), ha='center', fontsize=11,
                color='red', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.8))

# --- 无向图 ---
ax = axes[0]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.set_title('无向图 Undirected Graph', fontsize=14, fontweight='bold')
ax.axis('off')
edges_undir = [('A','B'), ('A','C'), ('B','C'), ('C','D'), ('C','E'), ('D','E')]
for e in edges_undir:
    draw_edge(ax, positions[e[0]], positions[e[1]])
for n, p in positions.items():
    draw_node(ax, n, p)

# --- 有向图 ---
ax = axes[1]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.set_title('有向图 Directed Graph', fontsize=14, fontweight='bold')
ax.axis('off')
edges_dir = [('A','B'), ('A','C'), ('B','C'), ('C','D'), ('E','C'), ('D','E')]
for e in edges_dir:
    p1 = positions[e[0]]
    p2 = positions[e[1]]
    dx, dy = p2[0]-p1[0], p2[1]-p1[1]
    dist = (dx**2+dy**2)**0.5
    # 缩短箭头，避免覆盖节点
    shrink = 0.09
    s1 = (p1[0]+shrink*dx/dist, p1[1]+shrink*dy/dist)
    s2 = (p2[0]-shrink*dx/dist, p2[1]-shrink*dy/dist)
    draw_edge(ax, s1, s2, directed=True)
for n, p in positions.items():
    draw_node(ax, n, p, color='#FF6B6B')

# --- 加权图 ---
ax = axes[2]
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.set_title('加权图 Weighted Graph', fontsize=14, fontweight='bold')
ax.axis('off')
edges_w = [('A','B',4), ('A','C',2), ('B','C',1), ('C','D',7), ('C','E',3), ('D','E',5)]
for e in edges_w:
    draw_edge(ax, positions[e[0]], positions[e[1]], weight=e[2])
for n, p in positions.items():
    draw_node(ax, n, p, color='#FFE66D')

plt.tight_layout()
plt.savefig('graph_types.png', dpi=100, bbox_inches='tight')
plt.show()


## 2. 图的表示方法

### 2.1 邻接矩阵 (Adjacency Matrix)

用一个 $n \times n$ 的二维数组表示图，其中 $n$ 是顶点数。

- `matrix[i][j] = 1` 表示顶点 $i$ 到顶点 $j$ 有边
- `matrix[i][j] = 0` 表示没有边
- 对于加权图，存储权重值而非1

**为什么用邻接矩阵？**
- 查询两个顶点是否相邻：$O(1)$ 时间！非常快
- 适合**稠密图**（边多的图）
- 缺点：空间 $O(n^2)$，对稀疏图浪费空间

### 2.2 邻接表 (Adjacency List)

每个顶点维护一个列表，存储它的所有邻居。

**为什么用邻接表？**
- 空间 $O(V + E)$，对稀疏图更高效
- 遍历某顶点的所有邻居更快
- 适合**稀疏图**（边少的图）

### 考试关键对比

| 操作 | 邻接矩阵 | 邻接表 |
|------|----------|--------|
| 空间复杂度 | $O(V^2)$ | $O(V+E)$ |
| 判断边是否存在 | $O(1)$ | $O(V)$ 最坏 |
| 遍历所有邻居 | $O(V)$ | $O(degree)$ |
| 适用场景 | 稠密图 | 稀疏图 |

In [ ]:
# === 邻接矩阵和邻接表的Python实现 ===

class GraphMatrix:
    '''用邻接矩阵表示的图 (Adjacency Matrix)'''
    def __init__(self, vertices):
        self.vertices = vertices
        self.n = len(vertices)
        self.v_index = {v: i for i, v in enumerate(vertices)}
        self.matrix = [[0] * self.n for _ in range(self.n)]

    def add_edge(self, v1, v2, weight=1, directed=False):
        i, j = self.v_index[v1], self.v_index[v2]
        self.matrix[i][j] = weight
        if not directed:
            self.matrix[j][i] = weight

    def display(self):
        header = '     ' + '  '.join(f'{v:>3}' for v in self.vertices)
        print(header)
        print('    ' + '-' * (len(self.vertices) * 5))
        for i, v in enumerate(self.vertices):
            row = f'{v:>3} |' + '  '.join(f'{self.matrix[i][j]:>3}' for j in range(self.n))
            print(row)


class GraphList:
    '''用邻接表表示的图 (Adjacency List)'''
    def __init__(self):
        self.adj = {}

    def add_vertex(self, v):
        if v not in self.adj:
            self.adj[v] = []

    def add_edge(self, v1, v2, weight=1, directed=False):
        self.add_vertex(v1)
        self.add_vertex(v2)
        self.adj[v1].append((v2, weight))
        if not directed:
            self.adj[v2].append((v1, weight))

    def display(self):
        for v in sorted(self.adj.keys()):
            neighbors = ', '.join(f'{n}({w})' for n, w in self.adj[v])
            print(f'{v} -> [{neighbors}]')


# 创建示例加权图
print('=== 邻接矩阵表示 (Adjacency Matrix) ===')
gm = GraphMatrix(['A', 'B', 'C', 'D', 'E'])
edges = [('A','B',4), ('A','C',2), ('B','C',1), ('C','D',7), ('C','E',3), ('D','E',5)]
for v1, v2, w in edges:
    gm.add_edge(v1, v2, w)
gm.display()

print()
print('=== 邻接表表示 (Adjacency List) ===')
gl = GraphList()
for v1, v2, w in edges:
    gl.add_edge(v1, v2, w)
gl.display()

print()
print('注意观察：同一张图的两种不同表示方式。')


In [ ]:
# === 可视化：邻接矩阵热力图 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 邻接矩阵热力图
vertices = ['A', 'B', 'C', 'D', 'E']
matrix_data = np.array(gm.matrix)
im = ax1.imshow(matrix_data, cmap='YlOrRd', interpolation='nearest')
ax1.set_xticks(range(len(vertices)))
ax1.set_yticks(range(len(vertices)))
ax1.set_xticklabels(vertices, fontsize=13)
ax1.set_yticklabels(vertices, fontsize=13)
ax1.set_title('邻接矩阵热力图\nAdjacency Matrix Heatmap', fontsize=14, fontweight='bold')

# 在每个格子中显示数值
for i in range(len(vertices)):
    for j in range(len(vertices)):
        text = ax1.text(j, i, matrix_data[i, j],
                       ha='center', va='center', fontsize=14, fontweight='bold',
                       color='white' if matrix_data[i, j] > 3 else 'black')

plt.colorbar(im, ax=ax1, label='权重 (Weight)')

# 邻接表可视化
ax2.axis('off')
ax2.set_title('邻接表可视化\nAdjacency List Visualization', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

y_positions = [0.85, 0.68, 0.51, 0.34, 0.17]
for idx, v in enumerate(sorted(gl.adj.keys())):
    y = y_positions[idx]
    # 顶点节点
    rect = plt.Rectangle((0.05, y-0.04), 0.08, 0.08, facecolor='#4ECDC4', edgecolor='black', lw=2)
    ax2.add_patch(rect)
    ax2.text(0.09, y, v, ha='center', va='center', fontsize=13, fontweight='bold')
    # 箭头和邻居
    x_start = 0.16
    for ni, (neighbor, weight) in enumerate(gl.adj[v]):
        x = x_start + ni * 0.17
        ax2.annotate('', xy=(x, y), xytext=(x-0.04, y),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
        rect2 = plt.Rectangle((x, y-0.04), 0.12, 0.08,
                              facecolor='#FFE66D', edgecolor='black', lw=1.5)
        ax2.add_patch(rect2)
        ax2.text(x+0.06, y, f'{neighbor}:{w}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()
print('左图：颜色越深表示权重越大。对角线为0（无自环）。')


## 3. 图遍历算法

### 3.1 深度优先搜索 DFS (Depth-First Search)

**核心思想：** 沿着一条路走到底，走不下去了再回头（回溯），尝试其他路。

**为什么叫"深度优先"？** 因为它总是优先往**更深的层次**探索，而不是先看同一层的其他节点。

**数据结构：** 使用**栈(Stack)**（或递归调用栈）

**DFS算法步骤：**
1. 将起始顶点压入栈，标记为已访问
2. 弹出栈顶顶点，处理它
3. 将该顶点的所有未访问邻居压入栈，标记为已访问
4. 重复步骤2-3直到栈为空

### 3.2 广度优先搜索 BFS (Breadth-First Search)

**核心思想：** 先访问所有距离为1的邻居，再访问距离为2的，以此类推，像水波纹一样扩散。

**为什么叫"广度优先"？** 因为它总是先把同一层（同一深度）的节点全部探索完，再进入下一层。

**数据结构：** 使用**队列(Queue)**

**BFS算法步骤：**
1. 将起始顶点加入队列，标记为已访问
2. 从队列头部取出一个顶点，处理它
3. 将该顶点的所有未访问邻居加入队列尾部，标记为已访问
4. 重复步骤2-3直到队列为空

### 关键区别

| 特性 | DFS | BFS |
|------|-----|-----|
| 数据结构 | 栈 (Stack) | 队列 (Queue) |
| 搜索方式 | 深入到底再回头 | 一层一层向外扩展 |
| 能找最短路径吗？| 不能 | 能（无权图）|
| 空间复杂度 | $O(V)$ | $O(V)$ |
| 时间复杂度 | $O(V+E)$ | $O(V+E)$ |

In [ ]:
# === DFS 和 BFS 实现 ===

def dfs(graph, start):
    '''深度优先搜索 - 使用栈实现'''
    visited = set()
    stack = [start]
    order = []
    steps = []  # 记录每一步的状态

    while stack:
        vertex = stack.pop()
        if vertex not in visited:
            visited.add(vertex)
            order.append(vertex)
            steps.append({
                'current': vertex,
                'visited': set(visited),
                'stack': list(stack),
                'action': f'访问 {vertex}'
            })
            # 逆序添加邻居（保证字母序优先访问）
            neighbors = sorted([n for n, w in graph.adj[vertex]], reverse=True)
            for neighbor in neighbors:
                if neighbor not in visited:
                    stack.append(neighbor)

    return order, steps


def bfs(graph, start):
    '''广度优先搜索 - 使用队列实现'''
    visited = set([start])
    queue = deque([start])
    order = []
    steps = []

    while queue:
        vertex = queue.popleft()
        order.append(vertex)
        steps.append({
            'current': vertex,
            'visited': set(visited),
            'queue': list(queue),
            'action': f'访问 {vertex}'
        })
        neighbors = sorted([n for n, w in graph.adj[vertex]])
        for neighbor in neighbors:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order, steps


# 测试
g = GraphList()
test_edges = [('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','F'), ('C','G')]
for v1, v2 in test_edges:
    g.add_edge(v1, v2)

print('=== 测试图的邻接表 ===')
g.display()
print()

dfs_order, dfs_steps = dfs(g, 'A')
bfs_order, bfs_steps = bfs(g, 'A')

print(f'DFS 遍历顺序: {" -> ".join(dfs_order)}')
print(f'BFS 遍历顺序: {" -> ".join(bfs_order)}')
print()
print('DFS像是走迷宫：一条路走到底，走不通再回头。')


In [ ]:
# === 可视化：DFS vs BFS 遍历过程对比 ===

# 树形图的位置
tree_pos = {
    'A': (0.5, 0.9),
    'B': (0.25, 0.6), 'C': (0.75, 0.6),
    'D': (0.12, 0.3), 'E': (0.38, 0.3),
    'F': (0.62, 0.3), 'G': (0.88, 0.3)
}
tree_edges = [('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','F'), ('C','G')]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('DFS vs BFS 遍历过程对比', fontsize=16, fontweight='bold')

def draw_tree_state(ax, pos, edges, visited, current, title, order_so_far):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=10)
    for v1, v2 in edges:
        ax.plot([pos[v1][0], pos[v2][0]], [pos[v1][1], pos[v2][1]],
                color='gray', lw=1.5, zorder=1)
    for node, p in pos.items():
        if node == current:
            color = '#FF6B6B'  # 当前正在访问 - 红色
        elif node in visited:
            color = '#4ECDC4'  # 已访问 - 绿色
        else:
            color = '#DDDDDD'  # 未访问 - 灰色
        circle = plt.Circle(p, 0.06, color=color, ec='black', lw=2, zorder=5)
        ax.add_patch(circle)
        ax.text(p[0], p[1], node, ha='center', va='center',
                fontsize=12, fontweight='bold', zorder=6)
    ax.text(0.5, 0.05, f'顺序: {" -> ".join(order_so_far)}',
            ha='center', fontsize=9, color='navy')

# DFS 过程 - 显示关键步骤
dfs_key_steps = []
visited_so_far = set()
order_so_far = []
for step in dfs_steps:
    visited_so_far = step['visited']
    order_so_far.append(step['current'])
    dfs_key_steps.append((set(visited_so_far), step['current'], list(order_so_far)))

# 选择4个关键帧
dfs_frames = [0, 1, 3, len(dfs_key_steps)-1]
for idx, frame in enumerate(dfs_frames):
    if frame < len(dfs_key_steps):
        v, c, o = dfs_key_steps[frame]
        draw_tree_state(axes[0][idx], tree_pos, tree_edges, v, c,
                       f'DFS Step {frame+1}: 访问{c}', o)

# BFS 过程
bfs_key_steps = []
visited_so_far = set()
order_so_far = []
for step in bfs_steps:
    visited_so_far = step['visited']
    order_so_far.append(step['current'])
    bfs_key_steps.append((set(visited_so_far), step['current'], list(order_so_far)))

bfs_frames = [0, 1, 3, len(bfs_key_steps)-1]
for idx, frame in enumerate(bfs_frames):
    if frame < len(bfs_key_steps):
        v, c, o = bfs_key_steps[frame]
        draw_tree_state(axes[1][idx], tree_pos, tree_edges, v, c,
                       f'BFS Step {frame+1}: 访问{c}', o)

# 添加图例
legend_elements = [
    mpatches.Patch(facecolor='#FF6B6B', edgecolor='black', label='当前访问 Current'),
    mpatches.Patch(facecolor='#4ECDC4', edgecolor='black', label='已访问 Visited'),
    mpatches.Patch(facecolor='#DDDDDD', edgecolor='black', label='未访问 Unvisited'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()
print('上排：DFS先深入到底（A->B->D...），再回溯')


## 4. Dijkstra最短路径算法

### 4.1 问题定义

给定一个**加权图**和一个**起点**，找到从起点到所有其他顶点的**最短路径**。

### 4.2 为什么需要Dijkstra？

- BFS只能处理**无权图**的最短路径
- 有了权重后，边数少的路径不一定最短（可能经过权重很大的边）
- Dijkstra专门解决**非负权重**图的单源最短路径问题

### 4.3 算法核心思想（贪心策略）

1. 初始化：起点距离为0，其他所有顶点距离为 $\infty$
2. 每次选择**当前距离最小**的未访问顶点 $u$
3. 对 $u$ 的每个邻居 $v$：如果经过 $u$ 到 $v$ 的距离更短，则更新 $v$ 的距离
4. 标记 $u$ 为已访问
5. 重复直到所有顶点都被访问

### 4.4 正确性直觉

**为什么贪心策略在这里是正确的？**

关键前提：所有边权重**非负**。

当我们选择距离最小的未访问顶点 $u$ 时，不可能有另一条路径经过其他未访问顶点到达 $u$ 更短——因为那些未访问顶点的距离都 $\geq$ $u$ 的距离，再加上非负边权只会更大。

如果有负权边，这个推理就不成立了！这就是Dijkstra不能处理负权边的原因。

In [ ]:
# === Dijkstra算法实现（带详细过程记录） ===

def dijkstra(graph, start):
    '''
    Dijkstra最短路径算法
    返回：距离字典、前驱字典、每步的详细状态
    '''
    # 初始化
    dist = {v: float('inf') for v in graph.adj}
    dist[start] = 0
    prev = {v: None for v in graph.adj}
    visited = set()
    # 优先队列：(距离, 顶点)
    pq = [(0, start)]
    steps = []

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)

        step_info = {
            'current': u,
            'dist_current': d,
            'visited': set(visited),
            'distances': dict(dist),
            'updates': []
        }

        for neighbor, weight in graph.adj[u]:
            if neighbor not in visited:
                new_dist = d + weight
                if new_dist < dist[neighbor]:
                    old_dist = dist[neighbor]
                    dist[neighbor] = new_dist
                    prev[neighbor] = u
                    heapq.heappush(pq, (new_dist, neighbor))
                    step_info['updates'].append(
                        f'{neighbor}: {old_dist} -> {new_dist} (经过{u})'
                    )

        step_info['distances'] = dict(dist)
        steps.append(step_info)

    return dist, prev, steps


def get_path(prev, target):
    '''从前驱字典中重建路径'''
    path = []
    current = target
    while current is not None:
        path.append(current)
        current = prev[current]
    return path[::-1]


# 创建测试图
g_dijkstra = GraphList()
dijkstra_edges = [
    ('A', 'B', 4), ('A', 'C', 2),
    ('B', 'C', 1), ('B', 'D', 5),
    ('C', 'D', 8), ('C', 'E', 10),
    ('D', 'E', 2), ('D', 'F', 6),
    ('E', 'F', 3)
]
for v1, v2, w in dijkstra_edges:
    g_dijkstra.add_edge(v1, v2, w)

dist, prev, steps = dijkstra(g_dijkstra, 'A')

print('=== Dijkstra算法详细过程 ===')
print(f'起点: A')
print()
for i, step in enumerate(steps):
    print(f'--- 第{i+1}步: 选择顶点 {step["current"]} (距离={step["dist_current"]}) ---')
    if step['updates']:
        print('  距离更新:')
        for upd in step['updates']:
            print(f'    {upd}')
    else:
        print('  无更新')
    dist_str = ', '.join(f'{k}:{v}' for k, v in sorted(step['distances'].items()))
    print(f'  当前距离表: {dist_str}')
    print()

print('=== 最终结果 ===')
for v in sorted(dist.keys()):
    if v != 'A':
        path = get_path(prev, v)


In [ ]:
# === 可视化：Dijkstra逐步求解过程 ===

dij_pos = {
    'A': (0.1, 0.5), 'B': (0.35, 0.8),
    'C': (0.35, 0.2), 'D': (0.6, 0.65),
    'E': (0.6, 0.35), 'F': (0.9, 0.5)
}

n_steps = len(steps)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Dijkstra算法逐步求解过程', fontsize=16, fontweight='bold')

for step_idx in range(min(6, n_steps)):
    ax = axes[step_idx // 3][step_idx % 3]
    step = steps[step_idx]
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'Step {step_idx+1}: 选择 {step["current"]} (dist={step["dist_current"]})',
                fontsize=12, fontweight='bold')

    # 画边
    for v1, v2, w in dijkstra_edges:
        p1, p2 = dij_pos[v1], dij_pos[v2]
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='#CCCCCC', lw=2, zorder=1)
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my+0.04, str(w), ha='center', fontsize=9, color='gray')

    # 画节点
    for node, pos in dij_pos.items():
        if node == step['current']:
            color = '#FF6B6B'
        elif node in step['visited']:
            color = '#4ECDC4'
        else:
            color = '#EEEEEE'
        circle = plt.Circle(pos, 0.06, color=color, ec='black', lw=2, zorder=5)
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], node, ha='center', va='center',
                fontsize=13, fontweight='bold', zorder=6)
        # 显示当前距离
        d = step['distances'][node]
        d_str = str(d) if d != float('inf') else 'INF'
        ax.text(pos[0], pos[1]-0.1, f'd={d_str}', ha='center', va='center',
                fontsize=9, color='navy', fontweight='bold')

# 如果步骤少于6，隐藏多余的子图
for step_idx in range(n_steps, 6):
    axes[step_idx // 3][step_idx % 3].axis('off')

legend_elements = [
    mpatches.Patch(facecolor='#FF6B6B', edgecolor='black', label='当前选中'),
    mpatches.Patch(facecolor='#4ECDC4', edgecolor='black', label='已确定最短路径'),
    mpatches.Patch(facecolor='#EEEEEE', edgecolor='black', label='未确定'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()
print('Dijkstra每一步都贪心地选择当前距离最小的未访问顶点。')


## 5. A*算法：更聪明的搜索

### 5.1 Dijkstra的局限

Dijkstra算法向**所有方向**均匀扩展——它不知道目标在哪里。如果你要从北京到上海，Dijkstra可能先花很多时间探索往西走的路径。

### 5.2 A*的核心改进

A*在Dijkstra的基础上加入了**启发式函数(Heuristic Function)**：

$$f(n) = g(n) + h(n)$$

- $g(n)$：从起点到 $n$ 的**实际距离**（和Dijkstra一样）
- $h(n)$：从 $n$ 到终点的**估计距离**（启发式）
- $f(n)$：经过 $n$ 的路径的**估计总代价**

A*每次选择 $f(n)$ 最小的顶点，而不是只看 $g(n)$。

### 5.3 启发式函数的要求

启发式函数必须**可接受(Admissible)**：永远不高估实际距离。

常用的启发式：
- **欧几里得距离**：直线距离，适合地图
- **曼哈顿距离**：适合网格地图

### 5.4 A* vs Dijkstra

| 特性 | Dijkstra | A* |
|------|----------|----|
| 搜索方向 | 无目标，向所有方向 | 有目标，倾向目标方向 |
| 评估函数 | $f(n) = g(n)$ | $f(n) = g(n) + h(n)$ |
| 效率 | 较低 | 通常更高 |
| 最优性 | 保证最优 | h可接受时保证最优 |
| 特殊关系 | - | h(n)=0时退化为Dijkstra |

In [ ]:
# === A*算法实现 ===

def heuristic(node, goal, positions):
    '''欧几里得距离作为启发式函数'''
    x1, y1 = positions[node]
    x2, y2 = positions[goal]
    return ((x1-x2)**2 + (y1-y2)**2) ** 0.5


def a_star(graph, start, goal, positions):
    '''A*算法'''
    open_set = [(0, start)]
    g_score = {v: float('inf') for v in graph.adj}
    g_score[start] = 0
    f_score = {v: float('inf') for v in graph.adj}
    f_score[start] = heuristic(start, goal, positions)
    prev = {v: None for v in graph.adj}
    visited = set()
    explored_order = []

    while open_set:
        _, current = heapq.heappop(open_set)
        if current in visited:
            continue
        visited.add(current)
        explored_order.append(current)

        if current == goal:
            break

        for neighbor, weight in graph.adj[current]:
            if neighbor not in visited:
                tentative_g = g_score[current] + weight
                if tentative_g < g_score[neighbor]:
                    g_score[neighbor] = tentative_g
                    f_score[neighbor] = tentative_g + heuristic(neighbor, goal, positions)
                    prev[neighbor] = current
                    heapq.heappush(open_set, (f_score[neighbor], neighbor))

    path = get_path(prev, goal)
    return path, g_score[goal], explored_order


def dijkstra_target(graph, start, goal):
    '''Dijkstra（到目标即停止，记录探索顺序）'''
    dist = {v: float('inf') for v in graph.adj}
    dist[start] = 0
    prev = {v: None for v in graph.adj}
    visited = set()
    pq = [(0, start)]
    explored_order = []

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)

        if u == goal:
            break

        for neighbor, weight in graph.adj[u]:
            if neighbor not in visited:
                new_dist = d + weight
                if new_dist < dist[neighbor]:
                    dist[neighbor] = new_dist
                    prev[neighbor] = u
                    heapq.heappush(pq, (new_dist, neighbor))

    path = get_path(prev, goal)
    return path, dist[goal], explored_order


# 创建一个网格图来对比
grid_graph = GraphList()
grid_pos = {}
rows, cols = 5, 7
for r in range(rows):
    for c in range(cols):
        node = f'{r},{c}'
        grid_pos[node] = (c, rows - 1 - r)  # y轴翻转
        grid_graph.add_vertex(node)
        if c + 1 < cols:
            grid_graph.add_edge(node, f'{r},{c+1}', weight=1)
        if r + 1 < rows:
            grid_graph.add_edge(node, f'{r+1},{c}', weight=1)

start_node = '0,0'
goal_node = '4,6'

# Dijkstra
dij_path, dij_dist, dij_explored = dijkstra_target(grid_graph, start_node, goal_node)
# A*
astar_path, astar_dist, astar_explored = a_star(grid_graph, start_node, goal_node, grid_pos)

print(f'Dijkstra: 探索了 {len(dij_explored)} 个节点, 路径长度={dij_dist}')
print(f'A*:       探索了 {len(astar_explored)} 个节点, 路径长度={astar_dist}')


In [ ]:
# === 可视化：Dijkstra vs A* 探索范围对比 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

def draw_grid_search(ax, title, explored, path, grid_pos, rows, cols):
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(-0.5, rows - 0.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    explored_set = set(explored)
    path_set = set(path)

    for r in range(rows):
        for c in range(cols):
            node = f'{r},{c}'
            x, y = grid_pos[node]
            if node in path_set:
                color = '#FF6B6B'
                alpha = 1.0
            elif node in explored_set:
                color = '#4ECDC4'
                alpha = 0.6
            else:
                color = '#EEEEEE'
                alpha = 0.5
            rect = plt.Rectangle((x-0.4, y-0.4), 0.8, 0.8,
                                facecolor=color, alpha=alpha,
                                edgecolor='gray', lw=0.5)
            ax.add_patch(rect)

    # 标记起点和终点
    sx, sy = grid_pos[start_node]
    gx, gy = grid_pos[goal_node]
    ax.text(sx, sy, 'S', ha='center', va='center', fontsize=16, fontweight='bold', color='white')
    ax.text(gx, gy, 'G', ha='center', va='center', fontsize=16, fontweight='bold', color='white')
    ax.text(cols/2, -0.4, f'探索节点数: {len(explored)}', ha='center', fontsize=12, color='navy')

draw_grid_search(ax1, 'Dijkstra (无方向感)', dij_explored, dij_path, grid_pos, rows, cols)
draw_grid_search(ax2, 'A* (有方向感)', astar_explored, astar_path, grid_pos, rows, cols)

legend_elements = [
    mpatches.Patch(facecolor='#FF6B6B', label='最短路径'),
    mpatches.Patch(facecolor='#4ECDC4', alpha=0.6, label='探索过的节点'),
    mpatches.Patch(facecolor='#EEEEEE', alpha=0.5, label='未探索的节点'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12)

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()
print('关键观察：两种算法找到的最短路径相同，但A*探索的节点更少！')
print('A*利用启发式函数"知道"目标大致在右下方，所以集中向那个方向搜索。')


## 6. 深入思考：Google地图用什么算法？

### 为什么真实导航不用简单的Dijkstra？

Google地图面对的是**数十亿个节点**的图（全球道路网络）。即使Dijkstra的时间复杂度是 $O((V+E)\log V)$，在这个规模下仍然太慢。

实际使用的技术：

1. **Contraction Hierarchies (收缩层次)**：预处理阶段建立"高速路优先"的层次结构，查询时大幅减少搜索空间

2. **A*变体**：用地理距离作为启发式，减少探索范围

3. **双向搜索**：同时从起点和终点搜索，在中间相遇

4. **图分区**：将大图分成区域，只精确计算跨区域的关键路径

5. **实时更新**：考虑交通状况、事故等动态因素

### 启示

A-Level考试要求你理解Dijkstra和A*的原理。而在真实工程中，这些算法是**基础积木**，工程师在此基础上组合、优化，才能处理实际规模的问题。理解基础算法的**原理**和**局限性**，才能知道如何改进。

---

## 7. 练习题 (Exercises)

### 练习1：基础概念

1. 用邻接矩阵和邻接表分别表示以下图：顶点{1,2,3,4}，边{(1,2), (1,3), (2,4), (3,4)}
2. 对于有100个顶点、200条边的图，邻接矩阵需要多少存储空间？邻接表呢？
3. 社交网络（10亿用户，平均每人300好友）应该用哪种表示？为什么？

### 练习2：DFS和BFS

对下面的图从顶点A开始：
- 写出DFS遍历顺序
- 写出BFS遍历顺序
- 用代码验证你的答案

### 练习3：Dijkstra

在下面的代码中手动跟踪Dijkstra算法的每一步。

### 练习4（拓展）：编程挑战

修改DFS代码，使其能够检测图中是否有环(Cycle)。

In [ ]:
# === 练习2：请完成DFS和BFS遍历 ===

# 构建练习用图
exercise_graph = GraphList()
exercise_edges = [
    ('A', 'B'), ('A', 'D'),
    ('B', 'C'), ('B', 'E'),
    ('C', 'F'),
    ('D', 'E'),
    ('E', 'F')
]
for v1, v2 in exercise_edges:
    exercise_graph.add_edge(v1, v2)

print('练习图的邻接表：')
exercise_graph.display()
print()


In [ ]:
# === 练习2答案（先自己尝试再看！）===

# 取消下面的注释来查看答案
# dfs_result, _ = dfs(exercise_graph, 'A')
# bfs_result, _ = bfs(exercise_graph, 'A')
# print(f'DFS: {" -> ".join(dfs_result)}')


In [ ]:
# === 练习3：手动跟踪Dijkstra ===

ex_graph = GraphList()
ex_edges = [
    ('S', 'A', 3), ('S', 'B', 5),
    ('A', 'B', 2), ('A', 'C', 6),
    ('B', 'C', 1), ('B', 'D', 4),
    ('C', 'D', 2)
]
for v1, v2, w in ex_edges:
    ex_graph.add_edge(v1, v2, w)

print('练习图（从S出发）：')
ex_graph.display()
print()
print('请在纸上画出这个图，然后手动执行Dijkstra算法。')
print('记录每一步选择的顶点和距离表的变化。')
print()


In [ ]:
# === 练习3答案 ===

# 取消注释查看
# dist3, prev3, steps3 = dijkstra(ex_graph, 'S')
# for i, step in enumerate(steps3):
#     print(f'Step {i+1}: 选择 {step["current"]}, 距离表: {step["distances"]}')
# print()
# for v in sorted(dist3.keys()):
#     if v != 'S':


In [ ]:
# === 练习4（拓展）：检测图中的环 ===

def has_cycle(graph):
    '''
    使用DFS检测无向图中是否有环
    提示：如果DFS过程中遇到一个已访问的节点（且不是父节点），则有环
    '''
    visited = set()

    def dfs_cycle(node, parent):
        visited.add(node)
        for neighbor, _ in graph.adj[node]:
            if neighbor not in visited:
                if dfs_cycle(neighbor, node):
                    return True
            elif neighbor != parent:
                return True  # 找到环！
        return False

    for vertex in graph.adj:
        if vertex not in visited:
            if dfs_cycle(vertex, None):
                return True
    return False


# 测试
# 有环的图
g_cycle = GraphList()
for v1, v2 in [('A','B'), ('B','C'), ('C','A')]:
    g_cycle.add_edge(v1, v2)
print(f'三角形图有环吗？{has_cycle(g_cycle)}')

# 无环的图（树）
g_tree = GraphList()
for v1, v2 in [('A','B'), ('A','C'), ('B','D')]:
    g_tree.add_edge(v1, v2)
print(f'树形图有环吗？{has_cycle(g_tree)}')

print()
print('思考：为什么检测环对于DFS来说是自然的？')


---

## 本节小结 (Summary)

| 概念 | 要点 |
|------|------|
| 图的类型 | 有向/无向、加权/无权 |
| 邻接矩阵 | $O(V^2)$空间，$O(1)$查询边，适合稠密图 |
| 邻接表 | $O(V+E)$空间，遍历邻居高效，适合稀疏图 |
| DFS | 栈/递归，深入到底再回溯，$O(V+E)$ |
| BFS | 队列，一层一层扩展，能找无权最短路径，$O(V+E)$ |
| Dijkstra | 贪心选最小距离，适用非负权图，$O((V+E)\log V)$ |
| A* | Dijkstra + 启发式，更高效地找目标最短路径 |

**下节预告：** 机器学习基础——从图算法到智能决策！